In [ ]:
import os

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [ ]:
!pip install plotly

In [ ]:
comments= pd.read_csv(r"E:\Data-Science-Projects\Project-1\UScomments.csv" , on_bad_lines="skip")

In [ ]:
comments

In [ ]:
comments.duplicated()

In [ ]:
comments[comments.duplicated(keep=False)].sort_values("comment_text")

In [ ]:
comments = comments.drop_duplicates()

In [ ]:
comments.isnull().sum()

In [ ]:
## drop means not present values means missing values

In [ ]:
import warnings
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
comments.dropna(inplace=True)     ##dropna removes missing value

In [ ]:
comments.isnull().sum()

### 2.Sentimental Analysis


In [ ]:
import sys
!{sys.executable} -m pip install nltk

In [ ]:
import nltk

In [ ]:
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia= SentimentIntensityAnalyzer()

In [ ]:
comments["comment_text"]

In [ ]:
sia.polarity_scores("MY FAN . attendance")['compound']

In [ ]:
sia.lexicon

In [ ]:
def analyze_comment_sentiment(comment):
    compound = sia.polarity_scores(comment)['compound']

    if compound>0.05:
        label="Positive "
        insight="the comment express positive emotion or approval"

    elif compound<0.05:
        label="Negative"
        insight="the comment shows dis.satisfaction or negative emotion"

    else:
        label="neutral"
        insight="the comment is informational or emotionally neutral"

    return {
    "label" : label ,
    "score" : compound ,
    "insight" : insight
    }

In [ ]:
analyze_comment_sentiment("MY FAN . attendance")

#### 3.Perform Emoji's Analysis


In [ ]:
import sys
!{sys.executable} -m pip install emoji

In [ ]:
import emoji

In [ ]:
comments["comment_text"].head(25)

In [ ]:
comment= "trending 😉"


In [ ]:
emoji.EMOJI_DATA

In [ ]:
emoji_list= []

for char in comment:
    if char in emoji.EMOJI_DATA:
        emoji_list.append(char)

In [ ]:
emoji_list

In [ ]:
all_emojis_list = []

for comment in comments["comment_text"].dropna():
    for char in comment:
        if char in emoji.EMOJI_DATA:
            all_emojis_list.append(char)

In [ ]:
all_emojis_list[0:10]

In [ ]:
len(all_emojis_list)

In [ ]:
from collections import Counter

In [ ]:
emojis_countn_list= Counter(all_emojis_list).most_common(10)

In [ ]:
emojis_countn_list

In [ ]:
emojis = [emoji for emoji , count in emojis_countn_list]

In [ ]:
counts= [count for emoji , count in emojis_countn_list]

In [ ]:
import sys
!{sys.executable} -m pip install -U nbformat

In [ ]:
px.bar(x= emojis ,
       y= counts ,
       title= " most used emojis in youtube comments" ,
       labels=
       {
           "x" : "Emoji",
           "y" : "count"
       })
       

#### 4.Collect Entire data of Youtube !

In [ ]:
import os

In [ ]:
files = os.listdir(r"E:\Data-Science-Projects\1.. Youtube Data Analysis-20260421T062557Z-3-001\1.. Youtube Data Analysis\additional_data")

In [ ]:
files

In [ ]:
files_csv = [file for file in files if '.csv' in file]

In [ ]:
files_csv

In [ ]:
full_df = pd.DataFrame()

path = r"E:\Data-Science-Projects\1.. Youtube Data Analysis-20260421T062557Z-3-001\1.. Youtube Data Analysis\additional_data"
for file in  files_csv:
    current_df = pd.read_csv(path+'/'+file , encoding="iso=8859-1" , on_bad_lines="skip")
    full_df = pd.concat([current_df , full_df] , ignore_index = True)

In [ ]:
full_df.shape

In [ ]:
full_df.head(3)

In [ ]:
full_df.columns

#### 5.How to export your data into (csv,json,db)

In [ ]:
full_df[full_df.duplicated()].shape

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
full_df.shape

In [ ]:
full_df[0:5000].to_csv(
    r"E:\Data-Science-Projects\1.. Youtube Data Analysis-20260421T062557Z-3-001\1.. Youtube Data Analysis.csv",
    index=False
)

In [ ]:
full_df[0:5000].to_json(r"E:\Data-Science-Projects\Exported_data/Youtube_sample.json")

In [ ]:
import sys
!{sys.executable} -m pip install sqlalchemy

In [ ]:
from sqlalchemy import create_engine

In [ ]:
engine = create_engine(r'sqlite:///E:\Data-Science-Projects\Exported_data/Youtube_data.sqlite')

In [ ]:
full_df.to_sql("Users", con=engine, if_exists="replace", index=False)

#### 6.. Which categorry dominates Youtube ?

In [ ]:
full_df.dtypes

In [ ]:
full_df["trending_date"]= pd.to_datetime(full_df["trending_date"] , format = "%y.%d.%m")

In [ ]:
full_df.dtypes

In [ ]:
import json


In [ ]:
path = r"E:\Data-Science-Projects\1.. Youtube Data Analysis-20260421T062557Z-3-001\1.. Youtube Data Analysis\additional_data\US_category_id.json"

In [ ]:
with open(path , 'r' , encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
data

In [ ]:
data["items"][0]

In [ ]:
data["items"][0]['snippet']['title']

In [ ]:
data["items"][0]['id']

In [ ]:
cat_dict = { }

for item in data["items"]:
    cat_dict[int(item['id'])]=item['snippet']['title']


In [ ]:
cat_dict

In [ ]:
full_df["category_name"] = full_df["category_id"].map(cat_dict)

In [ ]:
full_df.head(3)

In [ ]:
pivot_df = full_df.groupby(["trending_date" , "category_name"])["views"].sum().unstack(fill_value =0)

In [ ]:
area_chart = px.area(
    data_frame=pivot_df ,
    x = pivot_df.index ,
    y = pivot_df.columns ,
    title = "Trending Momentum Over time by Category"
)

In [ ]:
area_chart

In [ ]:
top_categories = full_df.groupby(["category_name"])["views"].sum().nlargest(6).index

In [ ]:
top_categories

In [ ]:
filtered_df = pivot_df[top_categories]

In [ ]:
area_chart

#### 7..Do Viral Videos Actually get engagment or not...?


In [ ]:
full_df.columns

In [ ]:
full_df["engagement_rate"]=(full_df["likes"]+full_df["comment_count"]) / full_df["views"]

In [ ]:
bubble_sample = full_df.sample(50000)

In [ ]:
bubble = px.scatter(full_df ,
           x= "views" ,
           y= "engagement_rate" ,
           size = "comment_count" ,
           color = "category_name" ,
           hover_name ="title" ,
           title= "Engagement bubble Map : Views vs Engagement rate" ,
           size_max=60 )        

In [ ]:
bubble

In [ ]:
bubble.update_xaxes(type="log")

#### 8.. Views vs Engagement : Inside YouTube's Algorithm

In [ ]:
full_df.columns

In [ ]:
category_metrics = full_df.groupby('category_name').agg(
                                    total_views = ("views", "sum") ,
                                    avg_engagement_efficiency = ("engagement_rate" , "mean") ,
                                    video_count = ("video_id" , "count")).reset_index()


In [ ]:
category_metrics

In [ ]:
category_metrics.columns

In [ ]:
treemap = px.treemap(
    category_metrics,
    path=["category_name"],
    values="total_views",
    color="avg_engagement_efficiency",
    color_continuous_scale="RdYlGn",
    title="Category Attention Share with Engagement Efficiency Overlay",
    hover_data={
        "total_views": ":,.0f",
        "avg_engagement_efficiency": ":.3f",   
        "video_count": True
    }
)

In [ ]:
treemap

#### 9.. Is the audience actually engaged ?


In [ ]:
full_df.columns

In [ ]:
full_df["engagement_rate"].describe()

In [ ]:
category_engagement_stats = (
    full_df.groupby('category_name')['engagement_rate']
    .describe()
    .reset_index()
)

In [ ]:
category_engagement_stats.sort_values("mean", ascending=False)

In [ ]:
box = px.box(full_df,
             x="category_name",
             y="engagement_rate",
             color="category_name",
             title="Audience Engagement by Category")

In [ ]:
box